# Notebook 03: Data Cleaning & Preprocessing

## Introduction
Before we can build and train machine learning models, we must process the raw data. This notebook documents and executes our automated preprocessing pipeline. The raw files contain several data quality issues, such as missing values, unparsed date strings, and inconsistent boolean formatting, which we address here.

## Objectives:
1. **Load Raw Data**: Read `train.csv`, `test.csv`, `stores.csv`, and `features.csv` into memory.
2. **Inspect for Missing Values**: Identify columns containing nulls.
3. **Standardize Holiday Column**: Convert the `IsHoliday` column to boolean values using `clean_holiday_column` to handle inconsistent string/integer inputs.
4. **Handle Missing Values**:
   - **MarkDown1-5**: Impute missing values with `0.0` since a missing value indicates no markdown promotion was active.
   - **CPI & Unemployment**: Impute missing metrics using store-grouped forward-fill (`ffill`) and backward-fill (`bfill`) to propagate the latest known macroeconomic trends.
5. **Merge Datasets**: Join `train` and `test` data with store metadata and macroeconomic features on store/date keys.
6. **Save Processed Data**: Save output files to `data/processed/` for downstream modeling.

In [1]:
import sys
import os
import pandas as pd

# Add the project root to python path to import our modules
sys.path.append(os.path.abspath('..'))

## 1. Load and Inspect Raw Data
First, we load the raw files and display their summaries to understand their schemas and confirm which columns have missing values.

In [2]:
from src import config
from src.data_loader import load_raw_data

train, test, stores, features = load_raw_data()

Loading raw data...


In [3]:
print("--- Train Data Shape ---", train.shape)
print(train.info())
print("\n--- Missing Values in Train ---")
print(train.isnull().sum())

--- Train Data Shape --- (421570, 5)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 421570 entries, 0 to 421569
Data columns (total 5 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   Store         421570 non-null  int64  
 1   Dept          421570 non-null  int64  
 2   Date          421570 non-null  object 
 3   Weekly_Sales  421570 non-null  float64
 4   IsHoliday     421570 non-null  bool   
dtypes: bool(1), float64(1), int64(2), object(1)
memory usage: 13.3+ MB
None

--- Missing Values in Train ---
Store           0
Dept            0
Date            0
Weekly_Sales    0
IsHoliday       0
dtype: int64


In [4]:
print("--- Features Data Shape ---", features.shape)
print(features.info())
print("\n--- Missing Values in Features ---")
print(features.isnull().sum())

--- Features Data Shape --- (8190, 12)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8190 entries, 0 to 8189
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Store         8190 non-null   int64  
 1   Date          8190 non-null   object 
 2   Temperature   8190 non-null   float64
 3   Fuel_Price    8190 non-null   float64
 4   MarkDown1     4032 non-null   float64
 5   MarkDown2     2921 non-null   float64
 6   MarkDown3     3613 non-null   float64
 7   MarkDown4     3464 non-null   float64
 8   MarkDown5     4050 non-null   float64
 9   CPI           7605 non-null   float64
 10  Unemployment  7605 non-null   float64
 11  IsHoliday     8190 non-null   bool   
dtypes: bool(1), float64(9), int64(1), object(1)
memory usage: 712.0+ KB
None

--- Missing Values in Features ---
Store              0
Date               0
Temperature        0
Fuel_Price         0
MarkDown1       4158
MarkDown2       5269
MarkDown3   

## 2. Execute the Cleaning Pipeline
We run the modular data cleaning function `clean_data()` imported from `src/preprocessing.py`. This function carries out all our preprocessing objectives (standardizing columns, handling missing values, and merging datasets) to produce clean, ready-to-model dataframes.

In [5]:
from src.preprocessing import clean_data

train_cleaned, test_cleaned = clean_data()

Loading raw data...


Parsing date columns...
Converting numeric columns...
Handling missing markdown values...
Handling missing CPI and Unemployment values...
Cleaning IsHoliday column...
Merging datasets...
Checking merged data...
Missing values after cleaning:
Store           0
Dept            0
Date            0
Weekly_Sales    0
IsHoliday       0
Type            0
Size            0
Temperature     0
Fuel_Price      0
MarkDown1       0
MarkDown2       0
MarkDown3       0
MarkDown4       0
MarkDown5       0
CPI             0
Unemployment    0
dtype: int64
Saving cleaned datasets...


Data cleaning complete!


## 3. Verify Cleaned Data
Finally, we inspect the resulting dataframes. We print their shapes and summaries to verify that dates are parsed correctly, data structures are aligned, and there are zero missing values left in the dataset.

In [6]:
print("--- Cleaned Train Shape ---", train_cleaned.shape)
print(train_cleaned.info())
print("\n--- Missing Values in Cleaned Train ---")
print(train_cleaned.isnull().sum())

--- Cleaned Train Shape --- (421570, 16)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 421570 entries, 0 to 421569
Data columns (total 16 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   Store         421570 non-null  int64         
 1   Dept          421570 non-null  int64         
 2   Date          421570 non-null  datetime64[ns]
 3   Weekly_Sales  421570 non-null  float64       
 4   IsHoliday     421570 non-null  bool          
 5   Type          421570 non-null  object        
 6   Size          421570 non-null  int64         
 7   Temperature   421570 non-null  float64       
 8   Fuel_Price    421570 non-null  float64       
 9   MarkDown1     421570 non-null  float64       
 10  MarkDown2     421570 non-null  float64       
 11  MarkDown3     421570 non-null  float64       
 12  MarkDown4     421570 non-null  float64       
 13  MarkDown5     421570 non-null  float64       
 14  CPI           421570 non-nu

In [7]:
train_cleaned.head()

,Store,Dept,Date,Weekly_Sales,IsHoliday,Type,Size,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment
0,1,1,2010-02-05,24924.50,False,A,151315,42.31,2.572,0.0,0.0,0.0,0.0,0.0,211.096358,8.106
1,1,1,2010-02-12,46039.49,True,A,151315,38.51,2.548,0.0,0.0,0.0,0.0,0.0,211.242170,8.106
2,1,1,2010-02-19,41595.55,False,A,151315,39.93,2.514,0.0,0.0,0.0,0.0,0.0,211.289143,8.106
3,1,1,2010-02-26,19403.54,False,A,151315,46.63,2.561,0.0,0.0,0.0,0.0,0.0,211.319643,8.106
4,1,1,2010-03-05,21827.90,False,A,151315,46.50,2.625,0.0,0.0,0.0,0.0,0.0,211.350143,8.106
